# Error propagation

## Tasks

Solve the tasks below.
For each task, give reasons for your solution by commenting in the notebook.
In conclusion, summarize your findings and contextualize them. What have you learned? Do the results make sense?

Your results will be examined for plagiarism. Please use your own plot styles, articulate your own thoughts, and present your own experimental approaches.

a) Propagate uncertainties for the following expressions using [SymPy](https://www.sympy.org) following the examples for [uncorrelated variables](https://nbviewer.jupyter.org/urls/www.physi.uni-heidelberg.de/Einrichtungen/FP/Datenanalyse/FP_Gaussian_error_propagation.ipynb?flush_cache=false) and [correlated variables](https://nbviewer.jupyter.org/urls/www.physi.uni-heidelberg.de/Einrichtungen/FP/Datenanalyse/FP_Gaussian_error_propagation_corr.ipynb?flush_cache=false) from the FP web page.

i) Find expressions for the absolute uncertainty $\sigma_z$ for $z = x + y$ and $z = x - y$ 

ii) Find expressions for the relative uncertainty $\sigma_z / z$ for $z = x \cdot y, \; z = x / y$ and $z = x^n y^n$

iii) The acceleration of gravity with a simple pendulum is given by the following formula:
$$g = 4  \pi^2 \frac{L}{T^2}$$
The relevant variables are the length $L$ of the pendulum and the period $T$ with the corresponding errors $\sigma_L$ and $\sigma_T$.

iv) The energy of a rotating object is given by:
$$E = \frac{1}{2} I \omega^2$$
The relevant variables are the angular velocity $\omega$ and the moment of inertia $I$ with the corresponding errors $\sigma_\omega$ and $\sigma_I$.

b) The radius $r$ and the height $h$ of a cylinder have been measured to $r = 5$ cm and $h = 2$ cm. The uncertainty for both measurements is $\sigma = 0.1$ cm. Determine the volume of the cylinder and its uncertainty assuming (i) that both measurements are uncorrelated and (ii) that both measurements are fully correlated.

c) The scattering angle and the radial distance of a certain particle can be determined from a position measurement $(x,y)$ 
$$r = \sqrt{x^2 + y^2}, \quad \theta = \mathrm{atan2}(y, x)$$
You find more on the [atan2](https://en.wikipedia.org/wiki/Atan2) function on wikipedia. The position ($x$,$y$) is measured with the corresponding uncertainties $\sigma_x$ and $\sigma_y$. Write a python function that returns the covariance matrix $U$ of $r$ and $\theta$ for a given covariance matrix $V$ of $x$ and $y$. Determine $U$ under the assumption that $x$ and $y$ are uncorrelated. Hint: The formulas you need can be found in the script.


## Solutions

In [1]:
import sympy as sp
import IPython.display as Id

def display(*args):
    text = " ".join(args)
    return Id.display(Id.Latex(text))

In this exercise, we make use of the Python library `SymPy` for symbolic mathematics.
`SymPy` lets the user define mathematical symbols, which can be combined into more complex expressions.
We will use `SymPy` to propagate uncertainties of various expressions.

We start with some input variables, $\vec x = (x_1, \dots, x_n)$, and their corresponding $n \times n$ covariance matrix,
$$
    \mathrm{\mathbf C}_{ij} = \mathrm{cov}(x_i, x_j) = \rho_{x_i x_j} \sigma_{x_i} \sigma_{x_j}.
$$
Now, we have some functions of these variables, $\vec f(\vec x) = (f_1(\vec x), \dots, f_m(\vec x))$, and their corresponding $m \times m$ covariance matrix,
$$
    \mathrm{\mathbf E}_{ij} = \mathrm{cov}(f_i, f_j) = \rho_{f_i f_j} \sigma_{f_i} \sigma_{f_j}.
$$
Using the (Jacobian) transition matrix,
$$
    \mathrm{\mathbf G}_{ij} = \frac{\partial f_i}{\partial x_j},
$$
we can calculate $\mathrm{\mathbf E}$ from $\mathrm{\mathbf C}$,
$$
    \mathrm{\mathbf E} = \mathrm{\mathbf G} \cdot \mathrm{\mathbf C} \cdot \mathrm{\mathbf G}^\mathsf T.
$$

We now implement two functions, `calc_E` and `calc_unc`.
The function `calc_E` uses the above equations to calculate $\mathrm{\mathbf E}$ given $\vec f$, $\vec x$, and $\mathrm{\mathbf C}$.
The function `calc_unc` uses `calc_E` to calculate the covariance matrix of a scalar function $f$.
It returns the propagated uncertainty, which is just the square root of the matrix's single element.

In [2]:
def calc_E(f: sp.Matrix, x: sp.Matrix, C: sp.Matrix) -> sp.Matrix:
    G = f.jacobian(x)
    E = G * C * G.T
    return E


def calc_unc(
    func: sp.Expr, args: sp.Matrix, cov: sp.Matrix, rel: bool = False
) -> sp.Expr:
    f = sp.Matrix([sp.log(func) if rel else func])
    E = calc_E(f, args, cov)[0, 0]
    return sp.sqrt(sp.expand(E))

### Propagating uncertainties
a) Propagate uncertainties for the following expressions.

i) Find expressions for the absolute uncertainty $\sigma_z$ for $z = x + y$ and $z = x - y$ 

ii) Find expressions for the relative uncertainty $\sigma_z / z$ for $z = x \cdot y$, $z = x / y$ and $z = x^n y^n$

iii) The acceleration of gravity with a simple pendulum is given by the following formula:
$$g = 4  \pi^2 \frac{L}{T^2}$$
The relevant variables are the length $L$ of the pendulum and the period $T$ with the corresponding errors $\sigma_L$ and $\sigma_T$.

iv) The energy of a rotating object is given by:
$$E = \frac{1}{2} I \omega^2$$
The relevant variables are the angular velocity $\omega$ and the moment of inertia $I$ with the corresponding errors $\sigma_\omega$ and $\sigma_I$.

In [3]:
x, y = sp.symbols("x, y", real=True)
sig_x, sig_y = sp.symbols("\\sigma_x, \\sigma_y", nonnegative=True)
rho = sp.symbols("\\rho", real=True)
n = sp.symbols("n", integer=True)

args = sp.Matrix([x, y])
cov = sp.Matrix(
    [[sig_x**2, rho * sig_x * sig_y], [rho * sig_x * sig_y, sig_y**2]]
)

In [4]:
z_1 = x + y
z_2 = x - y

sig_z_1 = calc_unc(z_1, args, cov)
sig_z_2 = calc_unc(z_2, args, cov)

temp = "For $z = {z}$, we find the absolute uncertainty $$ \\sigma_z = {sig_z}. $$"

display(
    temp.format(z=sp.latex(z_1), sig_z=sp.latex(sig_z_1)),
    temp.format(z=sp.latex(z_2), sig_z=sp.latex(sig_z_2)),
)

<IPython.core.display.Latex object>

In [5]:
z_3 = x * y
z_4 = x / y
z_5 = x**n * y**n

sig_z_3 = calc_unc(z_3, args, cov, rel=True)
sig_z_4 = calc_unc(z_4, args, cov, rel=True)
sig_z_5 = calc_unc(z_5, args, cov, rel=True)

temp = "For $z = {z}$, we find the relative uncertainty $$\\frac{{\\sigma_z}}{{z}} = {sig_z}.$$"

display(
    temp.format(z=sp.latex(z_3), sig_z=sp.latex(sig_z_3)),
    temp.format(z=sp.latex(z_4), sig_z=sp.latex(sig_z_4)),
    temp.format(z=sp.latex(z_5), sig_z=sp.latex(sig_z_5)),
)

<IPython.core.display.Latex object>

In [6]:
L, T, sig_L, sig_T = sp.symbols(
    "L, T, \\sigma_L, \\sigma_T", positive=True
)
args = sp.Matrix([L, T])
cov = sp.diag(sig_L**2, sig_T**2)

g = 4 * sp.pi**2 * L / T**2
sig_g = calc_unc(g, args, cov)
sig_g_rel = calc_unc(g, args, cov, rel=True)

temp1 = "For $g = {g}$, we find the absolute uncertainty $$ \\sigma_g = {sig_g}. $$"
temp2 = "Or, expressed via the relative uncertainty, we find $$ \\sigma_g = g {sig_g}. $$"

display(
    temp1.format(g=sp.latex(g), sig_g=sp.latex(sig_g)),
    temp2.format(g=sp.latex(g), sig_g=sp.latex(sig_g_rel)),
)

<IPython.core.display.Latex object>

In [7]:
I, omega, sig_I, sig_omega = sp.symbols(
    "I, \\omega, \\sigma_I, \\sigma_\\omega", positive=True
)
args = sp.Matrix([I, omega])
cov = sp.diag(sig_I**2, sig_omega**2)

E = I * omega**2 / 2
sig_E = calc_unc(E, args, cov)
sig_E_rel = calc_unc(E, args, cov, rel=True)

temp1 = "For $E = {E}$, we find the absolute uncertainty $$ \\sigma_E = {sig_E}. $$"
temp2 = "Or, expressed via the relative uncertainty, we find $$ \\sigma_E = E {sig_E}. $$"

display(
    temp1.format(E=sp.latex(E), sig_E=sp.latex(sig_E)),
    temp2.format(E=sp.latex(E), sig_E=sp.latex(sig_E_rel)),
)

<IPython.core.display.Latex object>

### Volume of a Cylinder
b) The radius $r$ and the height $h$ of a cylinder have been measured to $r = 5$ cm and $h = 2$ cm. 
The uncertainty for both measurements is $\sigma = 0.1$ cm. 
Determine the volume of the cylinder and its uncertainty assuming (i) that both measurements are uncorrelated and (ii) that both measurements are fully correlated.


We know the formula to be
$$
V = \pi r^2 h.
$$
If $r$ and $h$ are fully correlated, we know that $\rho = \pm 1$.

In [8]:
r, h, sig = sp.symbols("r, h, \\sigma", positive=True)
rho = sp.symbols("\\rho", real=True)

args = sp.Matrix([r, h])
cov_i = sp.diag(sig**2, sig**2)
cov_ii = sp.Matrix([[sig**2, rho * sig**2], [rho * sig**2, sig**2]])

V = sp.pi * r**2 * h
sig_V_i = calc_unc(V, args, cov_i)
sig_V_ii = calc_unc(V, args, cov_ii)

values = {r: 5, h: 2, sig: 0.1}

In [9]:
temp = "({V} \\pm {sig})\\,\\mathrm{{cm}}^3"
temp1 = "Under assumption (i), we find $$V = {res}.$$"
temp2 = "Under assumption (ii), we find $$V(\\rho = +1) = {plus},\\\\ V(\\rho = -1) = {minus}.$$"

display(
    temp1.format(
        res=temp.format(
            V=V.evalf(3, subs=values), 
            sig=sig_V_i.evalf(2, subs=values)
        )
    ),
    temp2.format(
        plus=temp.format(
            V=V.evalf(3, subs=values),
            sig=sig_V_ii.evalf(2, subs={**values, rho: 1}),
        ),
        minus=temp.format(
            V=V.evalf(4, subs=values),
            sig=sig_V_ii.evalf(2, subs={**values, rho: -1}),
        ),
    ),
)

<IPython.core.display.Latex object>

### Particle Position
c) The scattering angle and the radial distance of a certain particle can be determined from a position measurement $(x, y)$,
$$
    r = \sqrt{x^2 + y^2}, \quad \theta = \mathrm{atan2}(y, x).
$$
The position $(x, y)$ is measured with the corresponding uncertainties $\sigma_x$ and $\sigma_y$. 
Write a python function that returns the covariance matrix $\mathrm{\mathbf E}$ of $r$ and $\theta$ for a given covariance matrix $\mathrm{\mathbf C}$ of $x$ and $y$. 
Determine $\mathrm{\mathbf E}$ under the assumption that $x$ and $y$ are uncorrelated.

We have implemented the general function `calc_E`, which calculates the error matrix $\mathrm{\mathbf E}$, given $\vec f$, $\vec x$, and $\mathrm{\mathbf C}$.
Here, $\vec x = (x, y)$ and $\vec f = (r, \theta)$.
The covariance matrix $\mathrm{\mathbf C}$ of $\vec x$ is generally of the form
$$
    \mathrm{\mathbf C} = 
    \begin{pmatrix}
        \sigma_x^2 & \rho \sigma_x \sigma_y \\
        \rho \sigma_x \sigma_y & \sigma_y^2
    \end{pmatrix}.
$$
Here, $\sigma_x$ and $\sigma_y$ characterize the uncertainties of $x$ and $y$ respectively, while $\rho$ is their correlation coefficient.
We assume that $x$ and $y$ are uncorrelated, therefore $\rho = 0$.

In [10]:
x, y = sp.symbols("x, y", real=True)
r = sp.sqrt(x**2 + y**2)
theta = sp.atan2(y, x)

args = sp.Matrix([x, y])
f = sp.Matrix([r, theta])

sig_x, sig_y, rho = sp.symbols(
    "\\sigma_x, \\sigma_y, \\rho", positive=True
)
cov = sp.Matrix(
    [[sig_x**2, rho * sig_x * sig_y], [rho * sig_x * sig_y, sig_y**2]]
)

def func(C):
    return calc_E(f, args, C)

func(cov.subs({rho: 0}))

Matrix([
[             \sigma_x**2*x**2/(x**2 + y**2) + \sigma_y**2*y**2/(x**2 + y**2), -\sigma_x**2*x*y/(x**2 + y**2)**(3/2) + \sigma_y**2*x*y/(x**2 + y**2)**(3/2)],
[-\sigma_x**2*x*y/(x**2 + y**2)**(3/2) + \sigma_y**2*x*y/(x**2 + y**2)**(3/2),        \sigma_x**2*y**2/(x**2 + y**2)**2 + \sigma_y**2*x**2/(x**2 + y**2)**2]])